# Lab 2.1 · From Raw Posts to a Sentiment Baseline
**Chapter 2 · Preprocessing and Classical Sentiment**

**Goal:** build two cleaned text views, run **VADER** and a **TF-IDF + Logistic Regression** classifier, and explain their errors.
**Save for Chapter 3:** `outputs/posts_ch2.parquet` + your baseline scores.

📖 Interactive study guide: [Chapter 2](https://mehrdadjalali-ai.github.io/SocialMediaAnayltics/interactive/Chapter2_Preprocessing_Classical_Sentiment.html) — includes a live VADER explorer and a TF-IDF calculator.

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTD = ROOT / "data", ROOT / "outputs"
OUTD.mkdir(exist_ok=True)
SEED = 42
import re, pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix
try:
    import emoji
    count_emoji = emoji.emoji_count
    demojize = lambda t: emoji.demojize(t, delimiters=(" ", " "))
except ImportError:                      # fallback if the emoji package is missing
    EMO = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
    count_emoji = lambda t: len(EMO.findall(t))
    demojize = lambda t: EMO.sub(" emoji ", t)

## Walkthrough 1 · preprocess — never overwrite the raw text

In [2]:
df = pd.read_csv(DATA / "ev_posts.csv", parse_dates=["created_at"])
df["text_raw"] = df["text"]                       # never edited
df["mentions"] = df["text_raw"].str.findall(r"@\w+")
df["hashtags"] = df["text_raw"].str.findall(r"#\w+")
df["n_emoji"] = df["text_raw"].apply(count_emoji)
df["n_exclam"] = df["text_raw"].str.count("!")
df["n_caps"] = df["text_raw"].str.count(r"\b[A-Z]{2,}\b")

def clean_classical(t):
    t = re.sub(r"https?://\S+", " URL ", t)
    t = re.sub(r"@\w+", " USER ", t)
    t = demojize(t)
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)        # sooo -> soo
    return t.lower()

def clean_transformer(t):                      # light: keep emojis, case, punctuation
    t = re.sub(r"https?://\S+", "http", t)
    return re.sub(r"@\w+", "@user", t)

df["text_classical"] = df["text_raw"].apply(clean_classical)
df["text_transformer"] = df["text_raw"].apply(clean_transformer)
df[["text_raw", "text_classical", "text_transformer"]].head(3)

,text_raw,text_classical,text_transformer
0,The dealer says the next EV battery range upda...,the dealer says the next ev battery range upda...,The dealer says the next EV battery range upda...
1,"After 3 days, the EV charging has been better ...","after 3 days, the ev charging has been better ...","After 3 days, the EV charging has been better ..."
2,The dealer posted an update zu price for Monda...,the dealer posted an update zu price for monda...,The dealer posted an update zu price for Monda...


## Walkthrough 2 · leakage-safe split + two baselines

In [3]:
df = df.drop_duplicates("text_classical").reset_index(drop=True)   # before splitting!
idx_tr, idx_te = train_test_split(df.index, test_size=0.2, stratify=df["sentiment"], random_state=SEED)
df["split"] = "train"; df.loc[idx_te, "split"] = "test"
tr, te = df.loc[idx_tr], df.loc[idx_te]

vader = SentimentIntensityAnalyzer()
def vader_label(t):
    c = vader.polarity_scores(t)["compound"]
    return "positive" if c >= 0.05 else "negative" if c <= -0.05 else "neutral"
df["vader"] = df["text_raw"].apply(vader_label)     # raw text!

CUSTOM_SW = sorted(set(ENGLISH_STOP_WORDS) - {"not", "no", "nor", "never", "without", "cannot"})
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words=CUSTOM_SW),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(tr["text_classical"], tr["sentiment"])          # train only
df.loc[idx_te, "lr"] = model.predict(te["text_classical"])

for col in ["vader", "lr"]:
    print(col, round(f1_score(te["sentiment"], df.loc[idx_te, col], average="macro"), 3))
print(classification_report(te["sentiment"], df.loc[idx_te, "lr"]))

vader 0.594
lr 1.0
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       179
     neutral       1.00      1.00      1.00       170
    positive       1.00      1.00      1.00       124

    accuracy                           1.00       473
   macro avg       1.00      1.00      1.00       473
weighted avg       1.00      1.00      1.00       473



> ⚠️ **Too good to be true?** `ev_posts.csv` is synthetic and generated from a small set of sentence templates, so the classifier can reach an almost perfect macro-F1 by memorising template phrases. Treat that as a **spurious-feature warning sign**, not as a great model: inspect the top LR features (`model[-1].coef_`) and ask whether they would generalise. On real social media data expect macro-F1 around 0.7 (see the slides) — and always compare against VADER and the majority-class floor.

## Challenge 1 · Preprocessing

Compare the **default** scikit-learn stopword list with the custom list. How many posts still contain a negation word after each? Train the classifier with `stop_words='english'` and compare macro-F1 on the **same** split.

> 💡 **Hint:** `df['text_classical'].str.contains(r'\\b(not|no|never)\\b').sum()`

In [4]:
# Challenge 1 — your code here


**Your interpretation (Challenge 1):** _write 2–3 sentences — what did you find, and what could explain it?_

## Challenge 2 · VADER vs. classifier

Report macro-F1 for VADER, LR-unigrams and LR-uni+bigrams on the same test posts. Find **three posts** where VADER and LR disagree and explain each.

> 💡 **Hint:** `te_view = df.loc[idx_te]; te_view[te_view.vader != te_view.lr].sample(3, random_state=1)`

In [5]:
# Challenge 2 — your code here


**Your interpretation (Challenge 2):** _write 2–3 sentences — what did you find, and what could explain it?_

## Challenge 3 · Error analysis

Collect **five errors** of the LR model. For each: true, predicted, error type (sarcasm, negation scope, domain expression, spurious feature, unclear label, other language), source (text / preprocessing / model / label / scope) and **one targeted fix**.

> 💡 **Hint:** Start from the confusion matrix: `confusion_matrix(te['sentiment'], df.loc[idx_te,'lr'], labels=['negative','neutral','positive'])`.

In [6]:
# Challenge 3 — your code here


**Your interpretation (Challenge 3):** _write 2–3 sentences — what did you find, and what could explain it?_

## Save for Chapter 3

In [7]:
df.to_parquet(OUTD / "posts_ch2.parquet", index=False)
pd.DataFrame({"model": ["vader", "lr_uni_bi"],
              "macro_f1": [f1_score(te["sentiment"], df.loc[idx_te, c], average="macro") for c in ["vader", "lr"]]}
            ).to_csv(OUTD / "baseline_scores.csv", index=False)
print("saved outputs/posts_ch2.parquet")

saved outputs/posts_ch2.parquet
